# 미션 4-2 · 한국어 도메인 감성어 후보 탐색

> **목적:** KNU 한국어 감성사전에 추가할 **도메인 특화 단어 30개 이상**을 선정하기 위해, NSMC 영화 리뷰에서 자주 등장하는 형태소와 기존 사전의 매칭 상태를 조사한다.

- **입력:** `data/lexicons/ko/SentiWord_info.json`, `data/datasets/nsmc/ratings_train.txt`
- **분석기:** `KoNLPy.Okt.pos(norm=True, stem=True)`
- **출력:** 단어·품사별 빈도 CSV, KNU 매칭 결과 CSV, 후보 검토용 CSV (모두 `data/outputs/` 아래)
- **포함:** 등장 빈도 **두 기준**, 원본 사전·어간 정규화 후 매칭, TOP N 비교, 미등록 후보 탐색
- **제외:** 새로운 감성 점수의 자동 결정, 부정어/강조어 점수 계산, 모델 성능 평가

**집계 기준 (중요)**

- **중복 포함 (`token_count`):** 전체 리뷰에서 등장한 횟수를 모두 센다. 같은 리뷰에 3번 나오면 3회.
- **리뷰당 중복 제거 (`review_count`):** 한 리뷰에서는 1회만 센다. 같은 리뷰에 3번 나와도 1회.
- 따라서 `review_count`는 **단어가 등장한 서로 다른 리뷰의 개수(Document Frequency)**다. 전체 데이터셋에서 단어를 딱 한 번만 세는 방식은 순위를 매길 수 없어 사용하지 않는다.

**범위 주의**

- 도메인 특화 단어 후보를 선정할 때는 **NSMC train**만 사용한다.
- `ratings_test.txt`는 최종 성능 평가용으로 보존하고, 후보 단어를 고르는 데 사용하지 않는다.
- KNU의 기존 감성 점수는 **그 단어가 긍정·부정으로 해석된다는 검증된 정답이 아니라 사전 항목**이다.

## 0. 실행 환경 / 파일 경로

- `.venv`는 **프로젝트 루트**(`4-2/.venv`)에 존재해야 한다.
- 감성 사전과 원본 데이터는 모두 **프로젝트 루트의 `data/`** 아래에 있다.
- `notebooks/` 내부에서 실행하든 루트에서 실행하든 같은 경로를 탐색한다.
- 이 노트북은 **패키지를 설치하거나 데이터를 다운로드하지 않는다**.
- KoNLPy/Okt는 Java를 사용하므로, 필요하다면 `data/jdk-25/`의 JDK 경로를 커널 프로세스에 설정한다.

In [ ]:
from pathlib import Path
import csv
import json
import os
import platform
import sys
from collections import Counter, defaultdict
from html import escape

from IPython.display import HTML, display

# VS Code/Jupyter에서 현재 디렉터리가 root 또는 notebooks일 수 있다.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data").is_dir() and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("4-2/ 또는 4-2/notebooks/에서 실행해야 합니다.")
ROOT = ROOT.resolve()
DATA = ROOT / "data"
KNU_PATH = DATA / "lexicons" / "ko" / "SentiWord_info.json"
TRAIN_PATH = DATA / "datasets" / "nsmc" / "ratings_train.txt"
TEST_PATH = DATA / "datasets" / "nsmc" / "ratings_test.txt"
OUT_DIR = DATA / "outputs" / "sentiment_vocab"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 기존 설정 스크립트가 data/에 설치한 Java 25를 재사용한다.
# os.environ 수정은 노트북 커널의 현재 프로세스에만 적용된다.
manifest_path = DATA / "setup_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
java_candidates = [DATA / "jdk-25" / "Contents" / "Home", DATA / "jdk-25"]
if manifest.get("java_home"):
    java_candidates.append(Path(manifest["java_home"]))
if not os.environ.get("JAVA_HOME"):
    java_home = next((p for p in java_candidates if (p / "bin" / "java").exists()), None)
    if java_home:
        os.environ["JAVA_HOME"] = str(java_home)
        os.environ["PATH"] = str(java_home / "bin") + os.pathsep + os.environ.get("PATH", "")

for item in (KNU_PATH, TRAIN_PATH):
    if not item.is_file():
        raise FileNotFoundError(f"필요한 파일이 없습니다: {item}")

print("프로젝트 루트:", ROOT)
print("실행 Python  :", sys.executable)
print("플랫폼       :", platform.system(), platform.machine())
print("가상환경     :", sys.prefix)
print("JAVA_HOME    :", os.environ.get("JAVA_HOME", "미지정 (시스템 Java 사용)"))
print("출력 디렉터리:", OUT_DIR)
if Path(sys.prefix).resolve() != (ROOT / ".venv").resolve():
    print("⚠ 커널이 루트 .venv가 아닙니다. VS Code에서 .venv/bin/python을 선택하세요.")

## 1. 데이터 준비 상태 살펴보기

### KNU 한국어 감성사전

- `word`: 원본에 등록된 단어 또는 **여러 형태소로 이루어진 표현**
- `word_root`: 사전이 제공하는 어근 정보 (**Okt의 stem 결과와 동일하다고 가정하지 않는다**)
- `polarity`: 감성 점수 (`-2`, `-1`, `0`, `1`, `2`)
- **`0`점과 미등록(`None`)은 서로 다르다**.

### NSMC 리뷰 데이터

- `ratings_train.txt`: 후보 발굴에 사용할 데이터
- `ratings_test.txt`: 최종 평가용으로 보관
- 데이터의 `document`는 리뷰 원문이고, `label`은 영화 평가에 대한 긍정(1)/부정(0) 정답이다.
- **이번 탐색은 레이블을 학습하지 않는다.** 영화 리뷰 텍스트의 등장 표현만 집계한다.

In [ ]:
# KNU JSON 읽기: 파일 확장자가 json이므로 csv.read_csv를 사용하지 않는다.
with KNU_PATH.open("r", encoding="utf-8-sig") as fp:
    knu_rows = json.load(fp)
assert isinstance(knu_rows, list), "KNU 사전은 JSON 배열이어야 합니다."

print(f"KNU 전체 항목: {len(knu_rows):,}개")
print("KNU 점수 분포:", dict(sorted(Counter(int(row["polarity"]) for row in knu_rows).items())))
print("KNU 원본 앞 3개 (이모티콘 등도 포함 가능):")
for row in knu_rows[:3]:
    print(" -", row)

# NSMC는 TSV: 문장에 쉼표가 있어도 탭으로 구분되어 안전하게 처리된다.
def preview_nsmc(path, n=5):
    with path.open("r", encoding="utf-8", newline="") as fp:
        reader = csv.DictReader(fp, delimiter="\t")
        assert {"id", "document", "label"}.issubset(reader.fieldnames), reader.fieldnames
        return [row for _, row in zip(range(n), reader)]

print("\nNSMC train 샘플:")
for row in preview_nsmc(TRAIN_PATH, 3):
    print(f" - label={row['label']} | {row['document']}")
print("\nNSMC test 파일: ", "존재 (분석에서 미사용)" if TEST_PATH.exists() else "없음")

## 2. 형태소 분석기 관찰 — Okt + STEM

- **`norm=True`**: Okt의 텍스트 정규화 기능을 사용한다.
- **`stem=True`**: 동사·형용사 등 활용형을 기본형에 가깝게 정규화한다.
- **`pos()` 결과:** `(형태소 문자열, 품사)`의 나열.
- **중요:** `stem=True`가 모든 품사에 대해 동일하게 작동하거나, KNU `word`와 항상 일치하는 것은 아니다.
- **중요:** 동형이의어 혼동을 줄이기 위해 **형태소 문자열 + 품사** 조합으로 집계한다.

In [ ]:
from konlpy.tag import Okt

okt = Okt()
examples = [
    "영화가 정말 재미있었어요.",
    "내용이 너무 지루해서 지루하고 또 지루했어요.",
    "좋지는 않았지만 연기는 훌륭했습니다.",
]
for sentence in examples:
    print("\n문장:", sentence)
    print("원래 분석:", okt.pos(sentence, norm=True, stem=False))
    print("STEM 결과:", okt.pos(sentence, norm=True, stem=True))

## 3. 집계 기준과 처리 범위 설정

`MAX_REVIEWS = None`이면 **train 전체**를 처리한다. Intel Mac에서는 시간이 걸릴 수 있으므로, 최초 동작만 확인하고 싶다면 예를 들어 `MAX_REVIEWS = 3000`으로 바꿔도 된다.

- **토큰 출현 수:** 형태소가 나타날 때마다 1 증가한다.
- **출현 리뷰 수:** 한 리뷰에서 같은 `(형태소, 품사)`는 최대 1 증가한다.
- **집계 대상:** 조사·어미 등도 집계 데이터에 유지한다. 다만 결과를 탐색할 때는 감성 후보로 의미 있는 품사만 필터링할 수 있게 한다.
- **제외 대상:** 공백과 문장부호 품사(`Punctuation`)처럼 어휘 후보로 적절하지 않은 것만 제외한다.
- **표본 문장:** 후보 확인용으로 해당 단어가 등장한 첫 번째 리뷰를 저장한다. 감성 점수 추정에 자동 활용하지 않는다.

In [ ]:
# ===== 여기서 실험 범위를 조정 =====
MAX_REVIEWS = None  # 전체 train 사용. 테스트는 3000 등으로 바꿀 수 있다.
PROGRESS_EVERY = 10_000
SKIP_POS = {"Punctuation", "Space"}

# 단어+품사 단위의 Counter를 사용한다.
token_counts = Counter()       # 중복 허용: 전체 토큰 출현 횟수
review_counts = Counter()      # 리뷰당 1회: 해당 단어를 가진 리뷰 수
examples_by_token = {}         # 첫 출현 리뷰 (review_id, document)
processed_reviews = 0
empty_reviews = 0
invalid_rows = 0

with TRAIN_PATH.open("r", encoding="utf-8", newline="") as fp:
    reader = csv.DictReader(fp, delimiter="\t")
    for row in reader:
        if MAX_REVIEWS is not None and processed_reviews >= MAX_REVIEWS:
            break
        processed_reviews += 1
        sentence = (row.get("document") or "").strip()
        if not sentence:
            empty_reviews += 1
            continue

        # 예: [ ("영화", "Noun"), ("좋다", "Adjective"), ... ]
        # 전체 문장 안에서 같은 토큰이 반복되는 경우를 허용한다.
        try:
            tokens = [
                (form, pos)
                for form, pos in okt.pos(sentence, norm=True, stem=True)
                if form.strip() and pos not in SKIP_POS
            ]
        except Exception as err:
            invalid_rows += 1
            if invalid_rows <= 5:
                print("분석 실패 예시:", row.get("id"), type(err).__name__, str(err)[:120])
            continue

        token_counts.update(tokens)
        review_counts.update(set(tokens))  # 한 리뷰의 중복 단어를 제거
        for token in set(tokens):
            if token not in examples_by_token:
                examples_by_token[token] = (row.get("id", ""), sentence[:160])

        if PROGRESS_EVERY and processed_reviews % PROGRESS_EVERY == 0:
            print(f"처리: {processed_reviews:,} 리뷰 / 서로 다른 형태소·품사: {len(token_counts):,}")

print("\n== 집계 완료 ==")
print(f"처리 시도 리뷰 수: {processed_reviews:,}")
print(f"빈 리뷰: {empty_reviews:,} / 분석 실패: {invalid_rows:,}")
print(f"고유 (형태소, 품사) 조합 수: {len(token_counts):,}")
print(f"전체 토큰 출현 수(중복 포함): {sum(token_counts.values()):,}")

# 분석 실패가 있으면 기록을 확인하고 결과 해석 시 고려해야 한다.
if invalid_rows:
    print("⚠ 일부 문장은 분석되지 않았습니다. 해당 문장의 토큰은 집계에서 제외됐습니다.")

## 4. 집계 결과를 **독립된 CSV 파일**로 저장

- **이 파일만으로도 2번 작업을 재사용**할 수 있게 `token_count`, `review_count`를 함께 저장한다.
- 항목마다 `(stemmed_token, pos)`와 예시 문장이 포함된다.
- Excel에서 한글이 제대로 보이도록 `utf-8-sig`로 저장한다.
- 다른 분석을 할 때 KoNLPy를 다시 실행하지 않고 이 CSV를 로드하면 된다.
- `MAX_REVIEWS` 값도 파일명에 기록해 전체 집계와 부분 집계를 구분한다.

In [ ]:
scope_name = "full" if MAX_REVIEWS is None else f"first_{MAX_REVIEWS}"
COUNT_CSV = OUT_DIR / f"nsmc_train_okt_stem_counts_{scope_name}.csv"

count_rows = []
for (token, pos), count in token_counts.items():
    review_id, example = examples_by_token[(token, pos)]
    count_rows.append({
        "token": token,
        "pos": pos,
        "token_count": count,          # 전체 등장 횟수: 중복 포함
        "review_count": review_counts[(token, pos)], # 리뷰당 1회 카운트
        "sample_review_id": review_id,
        "sample_review": example,
    })
count_rows.sort(key=lambda r: (-r["token_count"], r["token"], r["pos"]))

COUNT_FIELDS = ["token", "pos", "token_count", "review_count", "sample_review_id", "sample_review"]
with COUNT_CSV.open("w", encoding="utf-8-sig", newline="") as fp:
    writer = csv.DictWriter(fp, fieldnames=COUNT_FIELDS)
    writer.writeheader()
    writer.writerows(count_rows)

print("저장 완료:", COUNT_CSV.relative_to(ROOT))
print("저장된 단어·품사 조합:", len(count_rows))

# 간단한 저장 확인: 열 헤더와 첫 2행을 읽는다.
with COUNT_CSV.open("r", encoding="utf-8-sig", newline="") as fp:
    reader = csv.DictReader(fp)
    print("CSV 열:", reader.fieldnames)
    print("첫 2행:", [next(reader, None) for _ in range(2)])

## 5. KNU 사전과 매칭하기 위한 색인 구성

KNU의 `word`와 Okt의 `stem=True` 출력은 **서로 다른 체계**이므로 다음 두 수준으로 비교한다.

1. **원본 직접 매칭:** Okt의 `(token)`이 KNU의 `word` 문자열과 정확히 일치하는가?
2. **단일 형태소 STEM 매칭:** KNU의 `word`를 Okt로 분석했을 때 **정확히 하나**의 `(형태소, 품사)`만 나온 경우, 해당 키가 리뷰의 키와 일치하는가?

매칭 상태를 구분한다.

- `exact`: 원본 KNU `word`와 직접 일치
- `stem`: 원본은 달라도, 단일 형태소의 STEM 결과와 일치
- `unmatched`: 현재 매칭 방식으로 찾지 못함. **반드시 사전에 없는 의미라는 뜻은 아니다.**
- `ambiguous`: 서로 다른 감성 점수 후보가 발견됨. 임의의 점수를 선택하지 않는다.

**의도적인 제한**

- KNU의 **여러 형태소로 된 표현**은 이번 *단일 토큰 빈도 분석*과 별개이므로 자동 결합하지 않는다.
- `word_root` 필드는 별도 검증 없이 바로 매칭 키로 사용하지 않는다.
- 이 결과는 **후보 선별용**이며, 최종 감성 분석기의 정확한 매칭률과 동일하지 않다.

In [ ]:
# 같은 원본 표현에 여러 점수가 붙는 상황도 유지한다.
raw_lookup = defaultdict(lambda: {"scores": set(), "words": set()})
stem_lookup = defaultdict(lambda: {"scores": set(), "words": set()})
skipped_multi_token = 0
skipped_parse_error = 0

for entry in knu_rows:
    word = str(entry.get("word", "")).strip()
    if not word:
        continue
    polarity = int(entry["polarity"])
    raw_lookup[word]["scores"].add(polarity)
    raw_lookup[word]["words"].add(word)

    try:
        parts = [(form, pos) for form, pos in okt.pos(word, norm=True, stem=True)
                 if form.strip() and pos not in SKIP_POS]
    except Exception:
        skipped_parse_error += 1
        continue

    # 하나의 형태소로 분해되는 사전 항목에 한해 단일 토큰 색인에 등록.
    if len(parts) != 1:
        skipped_multi_token += 1
        continue
    key = parts[0]
    stem_lookup[key]["scores"].add(polarity)
    stem_lookup[key]["words"].add(word)

print("KNU 원본 문자열 색인:", len(raw_lookup), "개")
print("KNU 단일 형태소 STEM 색인:", len(stem_lookup), "개")
print("다중 형태소/비어있는 분석으로 제외:", skipped_multi_token)
print("형태소 분석 실패로 제외:", skipped_parse_error)


def knu_match(token, pos):
    """정확/단일형태소-STEM 순서로 후보 점수를 찾고, 모호한 경우는 미확정 처리."""
    if token in raw_lookup:
        item = raw_lookup[token]
        method = "exact"
    elif (token, pos) in stem_lookup:
        item = stem_lookup[(token, pos)]
        method = "stem"
    else:
        return {"knu_score": None, "match_status": "unmatched", "knu_words": "", "score_candidates": ""}

    scores = sorted(item["scores"])
    words = sorted(item["words"])
    if len(scores) != 1:
        return {
            "knu_score": None,
            "match_status": "ambiguous",
            "knu_words": "; ".join(words[:8]),
            "score_candidates": ",".join(map(str, scores)),
        }
    return {
        "knu_score": scores[0],    # 점수가 0이면 그대로 0; 매칭 실패 None과 다르다.
        "match_status": method,
        "knu_words": "; ".join(words[:8]),
        "score_candidates": str(scores[0]),
    }

# 간단한 함수의 경계조건: 미등록과 0점 구분을 확인한다.
assert knu_match("존재할리없는문자열_123abc", "Noun")["knu_score"] is None

## 6. 빈도 데이터 + KNU 감성 점수 결합 후 저장

- **저장되는 모든 행**은 앞서 생성한 단어·품사 집계 결과 한 행에 대응한다.
- `knu_score`는 정수 또는 `None`이며, `None`은 **미등록 또는 점수 충돌로 미확정**인 경우이다.
- 미등록인지 모호한지 판단하려면 `match_status`를 함께 확인한다.
- 분석 결과 CSV는 도메인 감성어 30개를 직접 검토할 때 근거 자료로 활용한다.

In [ ]:
ENRICHED_CSV = OUT_DIR / f"nsmc_train_okt_stem_knu_{scope_name}.csv"
matched_rows = []
for r in count_rows:
    matched_rows.append({**r, **knu_match(r["token"], r["pos"])})

ENRICHED_FIELDS = COUNT_FIELDS + ["knu_score", "match_status", "knu_words", "score_candidates"]
with ENRICHED_CSV.open("w", encoding="utf-8-sig", newline="") as fp:
    writer = csv.DictWriter(fp, fieldnames=ENRICHED_FIELDS)
    writer.writeheader()
    writer.writerows(matched_rows)

status_counts = Counter(r["match_status"] for r in matched_rows)
print("저장 완료:", ENRICHED_CSV.relative_to(ROOT))
print("매칭 상태(고유 형태소·품사 기준):", dict(status_counts))
print("감성점수 0인 항목 수:", sum(r["knu_score"] == 0 for r in matched_rows))
print("미등록(None) 항목 수:", status_counts["unmatched"])
print("다의적 점수(None) 항목 수:", status_counts["ambiguous"])

## 7. Top N 탐색 함수 만들기

아래 함수는 **같은 CSV 데이터**를 여러 조건으로 조회한다.

- **정렬:** `token_count` 또는 `review_count`
- **대상:** 전체 / KNU 미등록(`unmatched`) / 점수 모호(`ambiguous`)
- **품사 필터:** 전체 품사를 보거나, 내용어 중심으로 좁힐 수 있다.
- 결과에는 `KNU 점수`가 표시되며, 찾지 못하면 **문자 그대로 `None`**을 표시한다.

**해석 시 주의**

- 조사·어미 등도 많이 등장하므로, *도메인 감성어 후보*를 살필 때는 내용어 필터가 유용하다.
- 자주 등장한다는 이유만으로 긍정/부정 단어라고 단정하면 안 된다. 예: `영화`, `배우`는 빈번하더라도 그 자체로 감성 점수가 없을 수 있다.

In [ ]:
# 내용어 중심 필터. Okt가 부여하는 품사명 기준.
CONTENT_POS = {"Noun", "Verb", "Adjective", "Adverb", "Exclamation"}
TOP_N = 30
SELECTED_POS = CONTENT_POS  # 모든 품사를 보고 싶다면 None으로 변경


def select_top(rows, sort_by, n=TOP_N, status=None, allowed_pos=SELECTED_POS):
    """조건에 맞는 단어를 빈도 기준 내림차순으로 정렬한다."""
    assert sort_by in {"token_count", "review_count"}
    result = [
        r for r in rows
        if (status is None or r["match_status"] == status)
        and (allowed_pos is None or r["pos"] in allowed_pos)
    ]
    return sorted(result, key=lambda r: (-r[sort_by], r["token"], r["pos"]))[:n]


def show_table(rows, columns, title):
    """pandas 추가 설치 없이 Jupyter에서 읽기 쉬운 HTML 표를 표시한다."""
    print(f"{title} (표시 {len(rows)}개)")
    if not rows:
        print("조건에 맞는 단어가 없습니다.")
        return
    th = "".join(f"<th>{escape(label)}</th>" for _, label in columns)
    trs = []
    for row in rows:
        tds = "".join(
            "<td>" + escape(
                "None" if row.get(key) is None or row.get(key) == "" and key == "knu_score"
                else f"{row.get(key):,}" if isinstance(row.get(key), int) and key.endswith("_count")
                else str(row.get(key, ""))
            ) + "</td>"
            for key, _ in columns
        )
        trs.append(f"<tr>{tds}</tr>")
    display(HTML(
        "<div style='overflow-x:auto;max-height:720px'>"
        "<table style='border-collapse:collapse;width:100%'>"
        "<thead style='position:sticky;top:0;background:var(--jp-layout-color1,#fff)'>"
        f"<tr>{th}</tr></thead><tbody>{''.join(trs)}</tbody></table></div>"
    ))

TABLE_COLS = [
    ("token", "STEM 단어"), ("pos", "품사"),
    ("token_count", "전체 출현 수"), ("review_count", "출현 리뷰 수"),
    ("knu_score", "KNU 점수"), ("match_status", "매칭"),
]

## 8. 전체 단어 Top N — 두 가지 집계 방식 비교

아래 두 표는 **같은 자료**를 다르게 정렬한 결과다.

- **표 A:** 중복 포함, `token_count` 내림차순
- **표 B:** 리뷰당 중복 제거, `review_count` 내림차순
- 한 리뷰에서 같은 단어가 반복되는 경향이 강하다면 두 표의 순위가 달라질 수 있다.
- 각 행에는 **기존 KNU 감성 점수(또는 None)**를 표시한다.

In [ ]:
show_table(
    select_top(matched_rows, "token_count"), TABLE_COLS,
    f"[A] 전체 Top {TOP_N} · 중복 포함 (token_count)"
)
show_table(
    select_top(matched_rows, "review_count"), TABLE_COLS,
    f"[B] 전체 Top {TOP_N} · 리뷰당 중복 제거 (review_count)"
)

## 9. **KNU 미등록** 단어 Top N — 30개 후보 발굴

- `match_status == 'unmatched'`만 조회한다.
- **표 C:** 미등록 단어 중 전체 출현 횟수가 많은 순
- **표 D:** 미등록 단어 중 등장하는 리뷰 수가 많은 순
- 점수는 당연히 `None`이다. 다만 **여러 형태소로 등록된 KNU 표현이 단일 토큰 매칭에 걸리지 않은 경우도 존재**하므로, `None`을 곧바로 진짜 사전 누락으로 해석하면 안 된다.
- 빈도가 높은 단어라도 문맥에 따라 평가가 바뀌면 감성어로 등록하지 않는 것이 적절할 수 있다.

In [ ]:
show_table(
    select_top(matched_rows, "token_count", status="unmatched"), TABLE_COLS,
    f"[C] 미등록 Top {TOP_N} · 중복 포함 (token_count)"
)
show_table(
    select_top(matched_rows, "review_count", status="unmatched"), TABLE_COLS,
    f"[D] 미등록 Top {TOP_N} · 리뷰당 중복 제거 (review_count)"
)

## 10. 점수 충돌 및 단어 사용 맥락 확인

- `ambiguous`는 KNU 매칭 후보들의 감성 점수가 서로 다른 경우다. **임의로 덮어쓰지 않는다.**
- 후보를 선택하기 전 **실제 리뷰 예문을 함께 검토**한다.
- 아래 함수의 조건을 변경하면 특정 품사 또는 원하는 문자열을 직접 검색할 수 있다.

In [ ]:
show_table(
    select_top(matched_rows, "review_count", n=15, status="ambiguous"),
    TABLE_COLS + [("score_candidates", "가능 점수")],
    "KNU 매칭 점수 충돌 사례 (상위 15개)"
)


def find_tokens(text, rows=matched_rows, max_results=20):
    """원하는 글자가 포함된 형태소와 첫 출현 문맥을 보여준다."""
    result = sorted(
        [r for r in rows if text in r["token"]],
        key=lambda r: (-r["review_count"], r["token"])
    )[:max_results]
    show_table(
        result,
        TABLE_COLS + [("sample_review", "첫 출현 리뷰 (앞 160자)")],
        f"'{text}' 포함 검색"
    )

# 예시 검색: 다른 단어로 바꾸면서 사용해 보자.
find_tokens("연기")

## 11. **수작업 검토용 후보 CSV** 생성

빈도만으로 감성 점수를 자동 부여하지 않는다. 아래 CSV를 확인하면서 직접 후보를 선정한다.

- KNU 미등록 단어를 **출현 리뷰 수** 기준으로 정렬한다.
- 각 행에 사용 맥락(`sample_review`)을 제공한다.
- 사람이 작성할 칼럼은 비워 둔다: `selected`, `custom_score`, `reason`.
- **점수 0도 의미 있는 값**이다. `None`/빈칸과 구분하여 작성한다.
- 30개 이상 추가하려면 단어마다 실제 감성 표현인지, 단어/품사/활용형 중 무엇을 등록할지 검토한다.
- 같은 파일을 반복 실행하면 덮어쓰므로, **수작업 기록이 시작된 뒤에는 복사본을 만들어 편집**하는 것을 권장한다.

In [ ]:
CANDIDATE_N = 100
CANDIDATE_CSV = OUT_DIR / f"domain_candidates_top{CANDIDATE_N}_{scope_name}.csv"
candidates = select_top(
    matched_rows,
    "review_count",
    n=CANDIDATE_N,
    status="unmatched",
    allowed_pos=CONTENT_POS,
)
CANDIDATE_FIELDS = [
    "token", "pos", "token_count", "review_count", "knu_score",
    "match_status", "sample_review_id", "sample_review",
    "selected", "custom_score", "reason",
]
with CANDIDATE_CSV.open("w", encoding="utf-8-sig", newline="") as fp:
    writer = csv.DictWriter(fp, fieldnames=CANDIDATE_FIELDS)
    writer.writeheader()
    for row in candidates:
        writer.writerow({
            **{field: row.get(field, "") for field in CANDIDATE_FIELDS},
            "selected": "",       # 예: Y
            "custom_score": "",   # 예: -2, -1, 0, 1, 2
            "reason": "",         # 예: '배달 지연 불만 표현' 등 직접 기재
        })
print("후보 검토 파일:", CANDIDATE_CSV.relative_to(ROOT))
print("후보 수:", len(candidates))
show_table(candidates[:10], TABLE_COLS + [("sample_review", "첫 출현 예문")], "후보 상위 10개 미리보기")

## 12. 결과 정리 및 후속 작업

**저장 파일 (모두 `4-2/data/outputs/sentiment_vocab/`)**

1. `nsmc_train_okt_stem_counts_*.csv`
   - NSMC 형태소별 **중복 포함 횟수** 및 **리뷰당 중복 제거 횟수**
2. `nsmc_train_okt_stem_knu_*.csv`
   - 위 집계 + **KNU 점수 / 매칭 상태 / 원본 사전 표현**
3. `domain_candidates_top100_*.csv`
   - 사람이 **30개 이상 직접 선정할 후보** 및 검토 메모 칼럼

**후속 학습**

- 선별 후보에서 영화 리뷰 감성에 실제 기여하는 표현을 **최소 30개** 선정한다.
- 후보의 **부정/긍정 점수·품사·선정 근거**를 기록한다.
- 그 결과를 `custom_lexicon.json`으로 변환하여 기존 KNU와 **충돌 정책**을 정한 후 병합한다.
- 부정어·강조어 규칙을 적용하고, **NSMC test**에서 Accuracy/F1로 최종 평가한다.

**주의사항**

- 이번 매칭은 **단일 토큰 분석**이다. `매우 만족`, `마음에 들다` 같은 복합 감성 표현은 나중에 별도로 다뤄야 한다.
- `unmatched`는 '이번 인덱스로 매칭되지 않음'을 뜻할 뿐 'KNU에 완전히 없는 감성 표현'을 보증하지 않는다.
- 높은 빈도는 **중요한 후보를 찾는 기준**이지, 감성의 강도나 방향을 결정하는 근거는 아니다.
- 중립적 도메인 단어를 단지 30개 조건을 맞추기 위해 추가하지 않는다.